In [ ]:
import pandas as pd
import seaborn as sns
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print(torch.cuda.is_available())      # should print True
print(torch.cuda.get_device_name(0))

In [ ]:
PATH = "fall-2026-ecse-551-task-1"

train = pd.read_csv(PATH+"/train.csv")
test = pd.read_csv(PATH+"/test.csv")

Y_train = train["label_name"]
X_train = train["image_path"]
X_test = test["id"]

def clean_path(path) :
  clean_path = path[7:]
  clean_path = "archive/RIVAL10_Kaggle" + clean_path
  return clean_path

def clean_path_test(path):
  path = "archive/RIVAL10_Kaggle/test/test_images/" + path
  return path

X_train = X_train.apply(clean_path)
X_test = X_test.apply(clean_path_test)

print(X_train)
print(X_test)
print(Y_train)

# Checking stats about the data

In [ ]:
g = sns.countplot(Y_train)

Y_train.value_counts()

In [ ]:
def read_image(path) :
  return np.array(Image.open(PATH+"/"+path).convert("RGB"))


In [ ]:
(X_train == "").any()
#check no test has null

In [ ]:
print(read_image(X_train[0]).shape)
print(read_image(X_train[0]).max())
print(read_image(X_train[0]).min())

# Prepare Data for pytorch

In [ ]:
#Prepare Data
X_train = X_train.apply(read_image)
X_test = X_test.apply(read_image)

In [ ]:
encodings={
    "truck":0,
    "frog":1,
    "cat":2,
    "plane":3,
    "bird":4,
    "car":5,
    "equine":6,
    "ship":7,
    "deer":8,
    "dog":9,
}

reverse_encodings = {v: k for k, v in encodings.items()}

Y_train = Y_train.replace(encodings)

In [ ]:
X_train = torch.from_numpy(np.stack(X_train[:1000])).float().permute(0, 3, 1, 2)
print(Y_train)
Y_train = torch.from_numpy(Y_train[:1000].astype("int64").to_numpy())
train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=64, shuffle=True)

X_test = torch.from_numpy(np.stack(X_test)).float().permute(0, 3, 1, 2)

# Define Model

In [ ]:
class CNN(nn.Module) :
  def __init__(self) :
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11, padding=0, stride=4)
    #out channels = nombre de filtres
    #in channels = les trois couleurs
    # Question : ou sont passés les couleurs ?
    self.relu = nn.ReLU()
    self.pool = nn.MaxPool2d(2)
    self.flatten = nn.Flatten()
    self.linear = nn.LazyLinear(10)


  def forward(self, x) :
    x = self.conv1(x)
    x = self.relu(x)
    x = self.pool(x)
    x = self.flatten(x)
    x = self.linear(x)
    return x


# Train Model

In [ ]:
device="cuda"
model = CNN().to(device)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
    print(epoch, loss.item())

# Evaluate Model

In [ ]:
model.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(X_test), 64):
        xb = X_test[i:i+64].to(device)
        preds.append(model(xb).argmax(dim=1).cpu())
predictions = torch.cat(preds)

In [ ]:
print(predictions)